# Analysis of single-cell RNA-seq data from sympathetic neurons

David Oliveira, 2025

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import os
import matplotlib.pyplot as plt
import scipy as sp
import matplotlib.cm as mpl_cm
from cycler import cycler

sc.settings.set_figure_params(dpi=150, facecolor='white')
sc.settings.n_jobs = 30

# Palette
from matplotlib import pyplot as plt, cm as mpl_cm
from cycler import cycler
palette=cycler(color=mpl_cm.tab20.colors)

In [2]:
wd = '/home/davi/Bioinfo/ObesityLab/SympGanglia/'

In [ ]:
# Human stellate
# adata = sc.read_h5ad(wd + 'Public_datasets/Human/Stellate/HumanStellate_analysed.h5ad')
adata = sc.read_h5ad(wd + 'SympAtlas/Interspecies_neurons.h5ad')
adata

In [ ]:
sc.pl.embedding(adata, basis='topoPaCMAP', color=['TopOMetry clusters'], legend_loc='on data', legend_fontsize=10, title='')

In [6]:
adata = adata.raw.to_adata()
adata.raw = adata

## Coexpression analysis

In [7]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def calculate_coexpression_percentage(adata, gene1, gene2, threshold=0.5):
    # Calculate co-expression percentage
    gene1_expressed = adata.raw.to_adata()[:, gene1].X.toarray() > threshold
    coexpression_cells = (adata[:, gene1].X.toarray() > threshold) & (adata[:, gene2].X.toarray() > threshold)
    total_cells = adata.shape[0]
    coexpression_percentage = (coexpression_cells.sum() / gene1_expressed.sum()) * 100
    return coexpression_percentage

def plot_coexpression_percentage(adata, gene1, gene2, threshold=0.5, ax=None):
    # Calculate co-expression percentage
    coexpression_percentage = calculate_coexpression_percentage(adata, gene1, gene2, threshold)

    # Add a temporary column to adata.obs for coloring
    gene1_expressed = adata[:, gene1].X.toarray() > threshold
    gene2_expressed = adata[:, gene2].X.toarray() > threshold
    #gene2_coexpressed = gene1_expressed.raw.to_adata()[:, gene2].X.toarray() > threshold
    adata.obs['coexpression'] = pd.Categorical(
        ['Co-expression' if (x & y) else 'Single Gene Expression' if (x | y) else 'No Expression' for x, y in zip(gene1_expressed, gene2_expressed)]
    )

    # Define colors for each category
    cmap = {'Co-expression': '#ad0303', 'No Expression': 'grey', 'Single Gene Expression': '#0350ad'}

    # Plot using scanpy
    sc.pl.scatter(adata, x=gene1, y=gene2, color='coexpression', size=15, title=f'Co-expression of {gene1} and {gene2}', use_raw=True, ax=ax, palette=cmap.values(), show=False)

    # Remove gridlines
    ax.grid(False)

    # Add dotted grey lines at threshold value
    ax.axvline(x=threshold, color='grey', linestyle='--')
    ax.axhline(y=threshold, color='grey', linestyle='--')

    # Calculate and plot the percentage of cells expressing each gene
    gene1_expression_percentage = (adata.raw.to_adata()[:, gene1].X.toarray() > threshold).mean() * 100
    gene2_expression_percentage = (adata.raw.to_adata()[:, gene2].X.toarray() > threshold).mean() * 100
    
    # ax.legend(title=f'Gene Expression (Threshold={threshold})')
    # ax.legend(handles=[
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#0350ad', markersize=10, label=f'{gene1}+ ({gene1_expression_percentage:.2f}%)'),
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#0350ad', markersize=10, label=f'{gene2}+ ({gene2_expression_percentage:.2f}%)'),
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#ad0303', markersize=10, label=f'{gene2}+/{gene1}+ ({coexpression_percentage:.2f}%)'),
    # ], loc='best')



In [8]:
sc.pp.neighbors(adata, use_rep='X_DM with bw_adaptive', n_neighbors=10, metric='euclidean')

In [ ]:
sc.tl.leiden(adata, resolution=0.2)

In [ ]:
sc.pl.embedding(adata, basis='topoPaCMAP', color=['leiden'], legend_loc='on data', legend_fontsize=10, title='', palette='tab20')

In [ ]:
new_cluster_names = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13'
]

adata.rename_categories("leiden", new_cluster_names)

In [ ]:
sc.pl.embedding(adata, basis='topoPaCMAP', color=['leiden'], legend_loc='on data', legend_fontsize=10, title='', palette='tab20')

In [ ]:
calculate_coexpression_percentage(adata, 'Th', 'Adrb2')

In [ ]:
# Example usage:
fig, ax = plt.subplots()
plot_coexpression_percentage(adata, 'Th', 'Adrb2', threshold=0.1, ax=ax)
plt.show()


In [ ]:

# List of genes to plot
genes = ['Th', 'Adrb2', 'Adrb1', 'Adrb3']
cluster_colors = [
    '#1f77b4', '#aec7e8', '#ffbb78', '#98df8a', '#d62728', '#9467bd', '#8c564b',
    '#c49c94', '#f7b6d2', '#c7c7c7', '#bcbd22', '#17becf', '#9edae5'
]

# Function to create violin plots for each gene, separated by cluster
def custom_violin_plot(adata, genes, cluster_key, figsize=(12,6), raw=True):
    fig, axes = plt.subplots(1, len(genes), figsize=(figsize[0]*len(genes), figsize[1]), sharey=True)

    for idx, gene_name in enumerate(genes):
        if raw:
            if gene_name in adata.raw.var_names:
                expression_data = adata.raw.to_adata()[:, gene_name].X
            else:
                print(f"Gene {gene_name} not found in the dataset.")
                continue
        else:
            if gene_name in adata.var_names:
                expression_data = adata[:, gene_name].X
            else:
                print(f"Gene {gene_name} not found in the dataset.")
                continue

        # Convert sparse matrix to dense, if needed
        if sp.sparse.issparse(expression_data):
            expression_data = expression_data.toarray()

        expression_data = np.squeeze(expression_data)  # Reduce dimensions

        # Get cluster labels
        cluster_labels = adata.obs[cluster_key]

        # Create DataFrame for easier manipulation
        data = pd.DataFrame({
            'Expression': expression_data,
            'Cluster': cluster_labels
        })

        # Select current axis
        ax = axes[idx] if len(genes) > 1 else axes

        # Create dataset list for each cluster
        unique_clusters = sorted(adata.obs[cluster_key].unique(), key=lambda x: int(x))
        dataset = [data.loc[data['Cluster'] == label, 'Expression'] for label in unique_clusters]

        # Create violin plot
        parts = ax.violinplot(dataset=dataset, showmeans=False, showmedians=False, showextrema=False)

        # Assign colors to clusters
        color_map = {cluster: cluster_colors[i % len(cluster_colors)] for i, cluster in enumerate(unique_clusters)}

        # In the plot, apply the colors to the violins
        for pc, cluster in zip(parts['bodies'], unique_clusters):
            pc.set_facecolor(color_map[cluster])
            pc.set_edgecolor('black')
            pc.set_alpha(1)
            pc.set_linewidth(0.5)

        # Add the threshold line
        ax.axhline(y=0.3, linestyle='--', color='grey', linewidth=1, label="Expression Threshold (0.3)")
        ax.set_ylim(0, 4)
        ax.set_title(f'{gene_name}', fontsize=32)
        ax.set_xticks(range(1, len(unique_clusters) + 1))
        ax.set_xticklabels(unique_clusters, rotation=45, ha='right')  # Cluster labels on x-axis
        ax.grid(False)  # Remove grid lines
        if idx == 0:
            ax.set_ylabel('Expression level', fontsize=14)

    plt.tight_layout()
    plt.show()

# Usage example
custom_violin_plot(adata, genes = ['Th', 'Adrb2', 'Adrb1', 'Adrb3'], cluster_key='leiden', figsize=(8,6), raw=False)
